Data imports

In [80]:
import h5py
from scipy.interpolate import griddata
import pandas as pd
import numpy as np
import torch
import os
import sys
import inspect
import random
import zipfile

In [81]:
# Clone the NPS repository
if not os.path.exists('NPS'):
    !git clone https://github.com/llnl/NPS.git

# Add the repository path to Python's system path so we can import modules from it
nps_path = os.path.abspath('NPS')
if nps_path not in sys.path:
    sys.path.append(nps_path)

print("NPS Repository successfully cloned and added to path!")

NPS Repository successfully cloned and added to path!


In [82]:
from NPS.model.convnext import ConvNeXt
import NPS.model.convnext as cnext
from NPS.model.convnext import ConvNeXtIsotropic

In [83]:
# Unzip cases_new.zip if it hasn't been unzipped yet
if not os.path.exists('cases_new'):
    print("Extracting cases_new.zip...")
    with zipfile.ZipFile('cases_new.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete!")
else:
    print("cases_new folder already exists.")

cases_new folder already exists.


Get shared parameters for one specific example.

In [84]:
def parse_parameters(case_dir_or_id):
    """
    Parses 'parameters.prm' from a given case directory or case ID (e.g., 'lhs_000').
    Returns a dictionary of parsed parameter keys and values.
    """
    # If only case ID is provided, construct the path
    case_dir = get_case_dir(case_dir_or_id)
    prm_path = os.path.join(case_dir, 'parameters.prm')
    if not os.path.exists(prm_path):
        raise FileNotFoundError(f"Parameter file not found at: {prm_path}")

    params = {}
    with open(prm_path, 'r') as f:
        for line in f:
            # Strip comments and whitespace
            line = line.split('#')[0].strip()
            if not line:
                continue
            # Handle standard parameter assignments (e.g., key = value)
            if '=' in line:
                key, val = line.split('=', 1)
                key = key.strip()
                val = val.strip()
                # Try converting to float or int if possible
                try:
                    if '.' in val or 'e' in val.lower():
                        params[key] = float(val)
                    else:
                        params[key] = int(val)
                except ValueError:
                    params[key] = val # Keep as string if it can't be numeric
    return params

# Test the function on 'lhs_000'
try:
    parsed_params = parse_parameters('lhs_000')
    print("Parsed parameters for lhs_000:")
    for k, v in parsed_params.items():
        print(f"  {k}: {v}")
except Exception as e:
    print("Error parsing parameters:", e)

Parsed parameters for lhs_000:
  set Number of dimensions: 2
  set Domain size X: 500
  set Domain size Y: 500
  set Domain size Z: 500
  set Subdivisions X: 3
  set Subdivisions Y: 3
  set Subdivisions Z: 3
  set Refine factor: 6
  set Element degree: 3
  set Mesh adaptivity: true
  set Max refinement level: 6
  set Min refinement level: 0
  set Steps between remeshing operations: 250
  set Criterion type: VALUE
  set Value lower bound: -0.9999
  set Value upper bound: 0.9999
  set Time step: 0.04
  set Number of time steps: 7500
  set Boundary condition for variable u: DIRICHLET: -0.6015
  set Boundary condition for variable phi: NATURAL
  set Boundary condition for variable mu: NATURAL
  set Model constant D: 0.625, DOUBLE
  set Model constant W0: 1.0, DOUBLE
  set Model constant delta: 0.6015, DOUBLE
  set Model constant epsilonM: 0.047625, DOUBLE
  set Model constant theta0: 0.125, DOUBLE
  set Model constant mult: 4.0, DOUBLE
  set Output condition: EQUAL_SPACING
  set Number of 

In [92]:
# Helper function for resolving directories
def get_case_dir(case_dir_or_id):
    """
    Resolves and verifies the directory path for a given case ID (e.g., 'lhs_000')
    or absolute/relative case path.
    """
    import os

    if not case_dir_or_id.startswith('cases_new/cases/'):
        case_dir = os.path.join('cases_new', 'cases', case_dir_or_id)
    else:
        case_dir = case_dir_or_id

    if not os.path.isdir(case_dir):
        raise FileNotFoundError(f"Case directory not found at: {case_dir}")
    return case_dir


In [88]:
def get_example_args(case_dir_or_id):
    """
    Parses 'parameters.prm' from a given case directory or case ID (e.g., 'lhs_000').
    """
    parsed_params = parse_parameters(case_dir_or_id)
    delta_raw = parsed_params.get('set Model constant delta') or parsed_params.get('delta', 0.0)
    epsilon_raw = parsed_params.get('set Model constant epsilonM') or parsed_params.get('epsilonM', 0.0)
    D_raw = parsed_params.get('set Model constant D') or parsed_params.get('D', 0.0)

    delta = clean_param_value(delta_raw)
    epsilon_M = clean_param_value(epsilon_raw)
    D = clean_param_value(D_raw)

    return delta, epsilon_M, D

# Helper to convert parameter string with potential suffix like ", DOUBLE" to float
def clean_param_value(val):
    if isinstance(val, (int, float)):
        return float(val)
    val_str = str(val).split(',')[0].strip()
    return float(val_str)

Get phi and u data for one snapshot in one example.


In [89]:
def get_example_data(case_dir_or_id, time_idx):
    """
    Loads the phi and u field data from the snapshots group inside rom_snapshots.h5
    for a specified case and time index.
    """
    import h5py
    import os

    case_dir = get_case_dir(case_dir_or_id)
    snapshots_path = os.path.join(case_dir, 'rom_snapshots.h5')
    if not os.path.exists(snapshots_path):
        raise FileNotFoundError(f"Snapshots file not found at: {snapshots_path}")

    with h5py.File(snapshots_path, 'r') as f:
        # Accessing nested datasets within the 'snapshots' group
        phi = f['snapshots/phi'][time_idx]
        u = f['snapshots/u'][time_idx]

    return phi, u

# Test the function to confirm it works
try:
    test_phi, test_u = get_example_data('lhs_000', 0)
    print("Successfully loaded data!")
    print(f" - phi shape: {test_phi.shape}, dtype: {test_phi.dtype}")
    print(f" - u shape: {test_u.shape}, dtype: {test_u.dtype}")
except Exception as e:
    print("Error testing the updated function:", e)

Successfully loaded data!
 - phi shape: (36864,), dtype: float64
 - u shape: (36864,), dtype: float64


Interpolate one snapshot of phi and u data onto 256x256 grid.

In [90]:
def interpolate_snapshot_to_grid(case_dir_or_id, phi_data, u_data, grid_size=256):
    """
    Interpolates unstructured 1D phi and u snapshots onto a uniform grid_size x grid_size spatial grid.

    Parameters:
      case_dir_or_id: str, case ID (e.g., 'lhs_000') or directory path
      phi_data: np.ndarray, 1D array of phi values with shape (N,)
      u_data: np.ndarray, 1D array of u values with shape (N,)
      grid_size: int, size of the target uniform grid (default is 256)

    Returns:
      grid_phi: np.ndarray, 2D array of interpolated phi with shape (grid_size, grid_size)
      grid_u: np.ndarray, 2D array of interpolated u with shape (grid_size, grid_size)
    """
    import h5py
    import numpy as np
    from scipy.interpolate import griddata
    import os

    case_dir = get_case_dir(case_dir_or_id)
    mesh_path = os.path.join(case_dir, 'reference_mesh.h5')
    if not os.path.exists(mesh_path):
        raise FileNotFoundError(f"Mesh file not found at: {mesh_path}")

    # Read physical coordinates (DoF coordinates)
    with h5py.File(mesh_path, 'r') as f:
        points = f['reference_mesh/dof_coordinates'][
            :]  # Shape is (36864, 2)

    # The domain is assumed to be [0, 500] x [0, 500] based on parameters.prm metadata
    grid_x, grid_y = np.mgrid[
        0 : 500 : complex(0, grid_size),
        0 : 500 : complex(0, grid_size),
    ]

    # Perform linear interpolation onto the uniform 2D grid
    grid_phi = griddata(points, phi_data, (grid_x, grid_y), method='linear')
    grid_u = griddata(points, u_data, (grid_x, grid_y), method='linear')

    return grid_phi, grid_u

In [91]:
# Test interpolation function on lhs_000 (time index 0)
test_phi, test_u = get_example_data('lhs_000', 0)
grid_phi, grid_u = interpolate_snapshot_to_grid('lhs_000', test_phi, test_u)

print(f"Interpolated 2D grid shape: {grid_phi.shape}")
print(f"Number of NaN elements in grid_phi: {np.isnan(grid_phi).sum()}")
print(f"Number of NaN elements in grid_u: {np.isnan(grid_u).sum()}")


Interpolated 2D grid shape: (256, 256)
Number of NaN elements in grid_phi: 0
Number of NaN elements in grid_u: 0


Use existing functions to get ML input for a specific case and snapshot

In [ ]:
def get_ml_input_for_snapshot(case_dir_or_id, time_idx, grid_size=256):
    """
    Constructs a PyTorch tensor of shape (1, 5, grid_size, grid_size) to use as machine learning input.
    The 5 channels correspond to: [phi, u, delta, epsilon_M, D]
    """
    import torch
    import numpy as np

    # 1. Parse parameters
    delta, epsilon_M, D = get_example_args(case_dir_or_id)

    # 2. Load the unstructured phi and u snapshots
    phi, u = get_example_data(case_dir_or_id, time_idx)

    # 3. Interpolate onto uniform 2D grid
    grid_phi, grid_u = interpolate_snapshot_to_grid(case_dir_or_id, phi, u, grid_size=grid_size)

    # 4. Clean NaN values (handling any boundary extrapolation gaps with physical backgrounds)
    clean_phi = np.nan_to_num(grid_phi, nan=-1.0)
    clean_u = np.nan_to_num(grid_u, nan=float(-delta))

    # 5. Create constant 2D grids for parameters
    delta_grid = np.full((grid_size, grid_size), delta, dtype=np.float32)
    epsilon_grid = np.full((grid_size, grid_size), epsilon_M, dtype=np.float32)
    D_grid = np.full((grid_size, grid_size), D, dtype=np.float32)

    # 6. Convert to tensors and stack/cat along channel dimension
    phi_t = torch.tensor(clean_phi, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    u_t = torch.tensor(clean_u, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    delta_t = torch.tensor(delta_grid, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    epsilon_t = torch.tensor(epsilon_grid, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    D_t = torch.tensor(D_grid, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    ml_input = torch.cat([phi_t, u_t, delta_t, epsilon_t, D_t], dim=1)
    return ml_input

In [ ]:
# Verify the single-snapshot pipeline function on 'lhs_000'
test_ml_input = get_ml_input_for_snapshot('lhs_000', 0)
print("Constructed ML Input shape:", test_ml_input.shape)
assert test_ml_input.shape == (1, 5, 256, 256), "ML Input tensor has incorrect shape!"

Constructed ML Input shape: torch.Size([1, 5, 256, 256])


In [ ]:
def get_ml_sequence_for_case(case_dir_or_id, time_indices, grid_size=256, batch_dim=True):
    """
    Constructs a sequence of ML input tensors for multiple time steps.

    Parameters:
      case_dir_or_id: str, case ID (e.g., 'lhs_000') or directory path
      time_indices: list of int, list of snapshot indices to process
      grid_size: int, size of the target uniform grid (default is 256)
      batch_dim: bool, if True returns shape (1, seq_len, 5, grid_size, grid_size)
                     else shape (seq_len, 5, grid_size, grid_size)

    Returns:
      sequence_tensor: torch.Tensor
    """
    import torch

    tensors = []
    for t in time_indices:
        # get_ml_input_for_snapshot returns shape (1, 5, grid_size, grid_size)
        snap_tensor = get_ml_input_for_snapshot(case_dir_or_id, t, grid_size=grid_size)
        # Remove the batch dimension of 1 temporarily for stacking
        tensors.append(snap_tensor.squeeze(0))

    # Stack along a new sequence dimension (dim=0)
    sequence_tensor = torch.stack(tensors, dim=0)

    if batch_dim:
        sequence_tensor = sequence_tensor.unsqueeze(0)

    return sequence_tensor


In [ ]:
# Test processing a sequence of 3 time steps on case 'lhs_000'
test_time_indices = [0, 1, 2]
test_seq_tensor = get_ml_sequence_for_case('lhs_000', test_time_indices)

print("Constructed Sequence Tensor shape:", test_seq_tensor.shape)
assert test_seq_tensor.shape == (1, 3, 5, 256, 256), "Sequence tensor has incorrect shape!"


Constructed Sequence Tensor shape: torch.Size([1, 3, 5, 256, 256])


# Train-Test Split
Assign each case to training or testing, get ML input for each group

In [ ]:
seed = 539
random.seed(seed)

cases = list(range(100))
random.shuffle(cases)

train_cases = cases[:80]
validation_cases = cases[80:90]
test_cases = cases[90:]

10


In [94]:
import torch

train_x = torch.tensor([])
train_y = torch.tensor([])

for case_num in train_cases:
    print(case_num)
    # Since cases in cases_new only contain 21 snapshots (indices 0 to 20),
    # we set the input sequence to indices 0 to 19 (range(20))
    # and the corresponding targets to indices 1 to 20 (range(1, 21))
    case_x = get_ml_sequence_for_case(f'lhs_{case_num:03d}', range(20))
    # Output targets only contain the first 2 channels: [phi, u]
    case_y = get_ml_sequence_for_case(f'lhs_{case_num:03d}', range(1, 21))[:, :, :2]
    train_x = torch.cat((train_x, case_x), dim=0)
    train_y = torch.cat((train_y, case_y), dim=0)

print("Start validation set")
validation_x = torch.tensor([])
validation_y = torch.tensor([])
for case_num in validation_cases:
    print(case_num)
    case_x = get_ml_sequence_for_case(f'lhs_{case_num:03d}', range(20))
    case_y = get_ml_sequence_for_case(f'lhs_{case_num:03d}', range(1, 21))[:, :, :2]
    validation_x = torch.cat((validation_x, case_x), dim=0)
    validation_y = torch.cat((validation_y, case_y), dim=0)

print("Start test set")
test_x = torch.tensor([])
test_y = torch.tensor([])
for case_num in test_cases:
    print(case_num)
    case_x = get_ml_sequence_for_case(f'lhs_{case_num:03d}', range(2))
    case_y = get_ml_sequence_for_case(f'lhs_{case_num:03d}', range(1, 3))[:, :, :2]
    test_x = torch.cat((test_x, case_x), dim=0)
    test_y = torch.cat((test_y, case_y), dim=0)

print(test_x.shape)
print(test_y.shape)

Start test set
30
47
88
18
5
91
71
96
24
37
torch.Size([10, 2, 5, 256, 256])
torch.Size([10, 2, 2, 256, 256])


Initialize CNN with 5 input channels. We must combine batch and timestep dimensions together in our x data before giving it to the CNN.

In [99]:
# Initialize the spatial model to accept 5 input channels
multi_channel_model = ConvNeXtIsotropic(
    in_chans=5,       # 5 channels: [phi, u, delta, epsilon_M, D]
    num_classes=2,    # 2 output channels: predicted [phi_next, u_next]
    depth=4,
    dim=64,
    DIM=2
)


In [100]:
### Example of flattening / unflattening data for CNN

# Get original shape dimensions
B, T, C, H, W = test_x.shape
print(f"Original 5D Input Shape: {test_x.shape}")

# Step 1: Flatten batch (B) and sequence (T) dimensions together
# This creates a 4D tensor with shape: [B * T, C, H, W]
flattened_input = test_x.view(B * T, C, H, W)
print(f"Flattened 4D Input Shape: {flattened_input.shape}")

# Step 2: Pass the flattened 4D tensor through the spatial model
multi_channel_model.eval()
with torch.no_grad():
    flattened_output = multi_channel_model(flattened_input)
print(f"Model 4D Output Shape:    {flattened_output.shape}")

# Step 3: Unflatten back to separate batch (B) and sequence (T) dimensions
# The model produces 2 output channels: [phi, u]
C_out = flattened_output.shape[1]
final_output = flattened_output.view(B, T, C_out, H, W)
print(f"Restored 5D Output Shape:  {final_output.shape}")

Original 5D Input Shape: torch.Size([10, 2, 5, 256, 256])
Flattened 4D Input Shape: torch.Size([20, 5, 256, 256])
Model 4D Output Shape:    torch.Size([20, 2, 256, 256])
Restored 5D Output Shape:  torch.Size([10, 2, 2, 256, 256])
